# 05 — Leukemia arms versus control

This notebook compares each leukemia arm with Empty_control, across the whole arm and within clusters.
Each arm is one library of pooled replicates, so no test across biological replicates is possible.
Resampling cells describes sampling stability rather than biological replication.

Steps:
1. Normalize AML counts and identify clusters with enough cells for a comparison.
2. Calculate fold changes, detection and bootstrap stability for each leukemia arm versus control.
3. Find shared changes, repeat the comparisons within clusters and prepare the top shared genes for the
figure.

Outputs: whole-arm and within-cluster contrasts, shared genes and top-gene fold changes in `leukemia_vs_control/`.

In [ ]:
import anndata as ad
import numpy as np
import pandas as pd
from IPython.display import display
from scipy import sparse

from settings import AML_DATASET, ATLAS_FILE, RESULTS_DIR, SEED

## Settings

In [ ]:
OUTPUT_DIR = RESULTS_DIR / "leukemia_vs_control"  # contrast tables used by notebook 06
LEUKEMIA_ARMS = ["AML1-ETO", "RasV12"]           # this order also fixes the bootstrap draw sequence
CONTROL_ARM = "Empty_control"                  # display name of the control library
MIN_GROUP_CELLS = 30                           # cells required on each side of a within-cluster contrast
NORMALIZATION_TOTAL = 10_000                   # counts per cell after normalization (CP10k)

PSEUDOCOUNT = 0.1                              # added to means to make ratios finite when one mean is zero
MIN_ABS_LOG2FC = 1                             # at least a twofold increase or decrease
MIN_DETECTION = 0.10                           # detection fraction required on either side
MIN_DETECTED_CELLS = 10                        # detected cells required on either side
N_BOOTSTRAP = 100                              # cell resamples per side of each contrast

STABLE_FRACTION = 0.95                         # resamples that must retain the direction and effect threshold
N_TOP_GENES = 30                               # shared genes shown in the figure

## Load AML counts and normalize each cell
Each cell's counts are divided by its total and multiplied by 10,000 to obtain counts per 10,000 (CP10k).
The control arm is called Empty_control in all contrast labels.

In [ ]:
OUTPUT_DIR.mkdir(parents=True, exist_ok=True)

atlas = ad.read_h5ad(ATLAS_FILE, backed="r")
is_aml = atlas.obs["dataset"].eq(AML_DATASET).to_numpy()
aml = atlas[is_aml].to_memory()
atlas.file.close()

aml.obs["arm"] = aml.obs["condition"].astype(str).replace({"control": CONTROL_ARM})
aml.obs["cluster"] = aml.obs["published_label"].astype(str)

counts = sparse.csr_matrix(aml.layers["counts"], dtype=np.float64)
cell_totals = np.asarray(counts.sum(axis=1)).ravel()
cp10k = sparse.csr_matrix(counts.multiply(NORMALIZATION_TOTAL / cell_totals[:, None]))

symbols = aml.var["symbol"]
candidate_feature = (aml.var_names.str.startswith("FBgn") & ~symbols.str.startswith("mt:")).to_numpy()

## Show the comparison design
A within-cluster contrast requires at least 30 cells in control and in that leukemia arm.
The table marks clusters testable in at least one leukemia arm; whole-arm contrasts use all AML cells.

In [ ]:
design = pd.crosstab(aml.obs["cluster"], aml.obs["arm"])[LEUKEMIA_ARMS + [CONTROL_ARM]]
enough_control = design[CONTROL_ARM] >= MIN_GROUP_CELLS
enough_leukemia = (design[LEUKEMIA_ARMS] >= MIN_GROUP_CELLS).any(axis=1)
design["testable_within_cluster"] = enough_control & enough_leukemia
display(design)

## Resample cells for stability
One random generator supplies every contrast, in AML1-ETO then RasV12 order.
A bootstrap draws cells with replacement, so some cells appear more than once and others are left out.
Multinomial counts give each cell's multiplicity in 100 resamples. A stable change keeps its direction and
fold-change threshold in at least 95% of resamples; this does not measure variation between biological
replicates.

In [ ]:
rng = np.random.default_rng(SEED)


def bootstrap_means(rows):
    """Mean CP10k per gene in 100 cell resamples, using the shared generator."""
    block = cp10k[rows]
    probabilities = np.full(len(rows), 1 / len(rows))

    draws = rng.multinomial(len(rows), probabilities, size=N_BOOTSTRAP)

    return (sparse.csr_matrix(draws) @ block).toarray() / len(rows)

## Calculate fold changes and detection
A pseudocount of 0.1 is added to each mean CP10k before taking the log2 ratio. This avoids division by zero
when a gene has no detected expression on one side, while treating the two sides in the same way.
A gene is eligible if detected in ≥ 10% of either side and in ≥ 10 cells of either side.

In [ ]:
def contrast(treated, control, name):
    """Effect sizes, detection and bootstrap stability for treated versus control cells."""
    treated_rows = np.flatnonzero(treated)
    control_rows = np.flatnonzero(control)

    mean_treated = np.asarray(cp10k[treated_rows].mean(axis=0)).ravel()
    mean_control = np.asarray(cp10k[control_rows].mean(axis=0)).ravel()

    detected_treated = np.asarray((counts[treated_rows] > 0).sum(axis=0)).ravel()
    detected_control = np.asarray((counts[control_rows] > 0).sum(axis=0)).ravel()
    detection_treated = detected_treated / len(treated_rows)
    detection_control = detected_control / len(control_rows)

    log2fc = np.log2((mean_treated + PSEUDOCOUNT) / (mean_control + PSEUDOCOUNT))
    enough_fraction = np.maximum(detection_treated, detection_control) >= MIN_DETECTION
    enough_cells = np.maximum(detected_treated, detected_control) >= MIN_DETECTED_CELLS
    eligible = enough_fraction & enough_cells

    direction = np.select(
        [eligible & (log2fc >= MIN_ABS_LOG2FC), eligible & (log2fc <= -MIN_ABS_LOG2FC)],
        ["up", "down"],
        "unchanged",
    )

    # Draw treated resamples before control resamples to keep the shared random sequence.
    bootstrap_treated = bootstrap_means(treated_rows)
    bootstrap_control = bootstrap_means(control_rows)
    bootstrap_log2fc = np.log2((bootstrap_treated + PSEUDOCOUNT) / (bootstrap_control + PSEUDOCOUNT))

    holds_up = (bootstrap_log2fc >= MIN_ABS_LOG2FC).mean(axis=0)
    holds_down = (bootstrap_log2fc <= -MIN_ABS_LOG2FC).mean(axis=0)
    stability = np.select([direction == "up", direction == "down"], [holds_up, holds_down], np.nan)

    # Record all genes; calls exclude transgenes and mitochondrial genes.
    table = pd.DataFrame(
        {
            "contrast": name,
            "gene_id": aml.var_names,
            "symbol": symbols.to_numpy(),
            "n_treated": len(treated_rows),
            "n_control": len(control_rows),
            "mean_cp10k_treated": mean_treated,
            "mean_cp10k_control": mean_control,
            "log2fc": log2fc,
            "detection_treated": detection_treated,
            "detection_control": detection_control,
            "eligible": eligible,
            "direction": direction,
            "bootstrap_stability": stability,
            "stable": stability >= STABLE_FRACTION,
            "candidate_feature": candidate_feature,
        },
    )

    table["called"] = table["direction"].ne("unchanged") & table["stable"] & table["candidate_feature"]

    return table

## Compare each whole leukemia arm with control
A change is called if |log2FC| ≥ 1 and the same direction and threshold hold in ≥ 95% of resamples.
AML1-ETO is tested before RasV12 so the bootstrap draws remain reproducible.

In [ ]:
arm = aml.obs["arm"].to_numpy()
whole = {}

for name in LEUKEMIA_ARMS:
    whole[name] = contrast(arm == name, arm == CONTROL_ARM, f"{name}_vs_{CONTROL_ARM}")

pd.concat(whole.values()).to_csv(OUTPUT_DIR / "whole_arm_contrasts.tsv.gz", sep="\t", index=False)

## Keep genes called in the same direction in both arms
Shared genes must be stable calls in both independent leukemia libraries.
Their direction and the two whole-arm log2 fold changes are saved in atlas gene order.

In [ ]:
aml1eto = whole["AML1-ETO"].set_index("gene_id")
rasv12 = whole["RasV12"].set_index("gene_id")

is_shared = aml1eto["called"] & rasv12["called"] & aml1eto["direction"].eq(rasv12["direction"])

shared = pd.DataFrame(
    {
        "symbol": symbols,
        "direction": aml1eto["direction"],
        "log2fc_AML1-ETO": aml1eto["log2fc"],
        "log2fc_RasV12": rasv12["log2fc"],
    },
).loc[is_shared]

shared.index.name = "gene_id"
shared.to_csv(OUTPUT_DIR / "shared_genes.tsv", sep="\t")
display(shared.head(10))

print(
    f"Shared leukemia genes: {shared['direction'].eq('up').sum()} up, "
    f"{shared['direction'].eq('down').sum()} down",
)

## Repeat the comparisons within testable clusters
The comparisons follow AML1-ETO then RasV12, with clusters in the design table's order.
These contrasts describe expression differences within a cluster rather than changes in composition.

In [ ]:
cluster = aml.obs["cluster"].to_numpy()
within = []

for name in LEUKEMIA_ARMS:

    for group in design.index[design["testable_within_cluster"]]:

        if design.loc[group, name] >= MIN_GROUP_CELLS:
            treated = (arm == name) & (cluster == group)
            control = (arm == CONTROL_ARM) & (cluster == group)

            contrast_name = f"{name}_vs_{CONTROL_ARM}|{group}"
            result = contrast(treated, control, contrast_name)

            within.append(result)

within = pd.concat(within, ignore_index=True)
contrast_parts = within["contrast"].str.split("|", expand=True)
within["arm_contrast"] = contrast_parts[0]
within["cluster"] = contrast_parts[1]
within.to_csv(OUTPUT_DIR / "within_cluster_contrasts.tsv.gz", sep="\t", index=False)

## Save the top 30 shared genes for the figure
Shared genes are ranked by the mean absolute log2 fold change of the two whole-arm contrasts.
Whole-arm columns come first, followed by within-cluster columns sorted by contrast name.

In [ ]:
shared_genes = shared.index
mean_absolute = (aml1eto.loc[shared_genes, "log2fc"].abs() + rasv12.loc[shared_genes, "log2fc"].abs()) / 2

ranked = aml1eto.loc[shared_genes].assign(mean_abs=mean_absolute).nlargest(N_TOP_GENES, "mean_abs").index

top_table = pd.DataFrame(
    {"whole AML1-ETO": aml1eto.loc[ranked, "log2fc"], "whole RasV12": rasv12.loc[ranked, "log2fc"]},
)

for contrast_name, rows in within.groupby("contrast"):
    column = contrast_name.replace(f"_vs_{CONTROL_ARM}|", " | ")
    top_table[column] = rows.set_index("gene_id").loc[ranked, "log2fc"]

top_table.index = symbols.loc[ranked]
top_table.to_csv(OUTPUT_DIR / "top_shared_genes_log2fc.tsv", sep="\t")
display(top_table.iloc[:10, :6])